In [1]:
import os
import json
import numpy as np
import pandas as pd

def readInCsv(filename, folderName):
    return pd.read_csv(f"{folderName}/{filename}", parse_dates=["Date"], index_col="Date")

def loadJson(filename):
    with open(filename, "r") as f:
        return json.load(f)

In [2]:
# DATEN EINLESEN & ANALYSEEINHEITEN BILDEN

In [2]:
def createYearSeperation(folderName):
    ALLAssetsYearSeperated = {}
    cleanData = os.listdir(folderName)
    
    for filename in cleanData:

        assetYearSeperated = {}
        assetName = filename.split("_")[0]

        df = readInCsv(filename, folderName)
        years = df.index.year.unique()

        for year in years:
            yearDF = df[df.index.year == year]
            assetYearSeperated[year] = yearDF

        ALLAssetsYearSeperated[assetName] = assetYearSeperated
        
    return ALLAssetsYearSeperated

dictionaryAssetYearData = createYearSeperation(folderName = "datenCLEAN")
dictionaryAssetYearData.keys()

dict_keys(['AAPL', 'DTE.DE', 'GDAXI', 'GSPC', 'JPM', 'NDX', 'PG', 'SIE.DE', 'XOM'])

In [3]:
# Reaktionsschwelle - Reaktionsfenster - Standardabweichung - Metadaten einlesen

In [4]:
reaktionsschwelle = loadJson("analyseparameter/reaktionsschwelle.json")
reaktionsfenster = loadJson("analyseparameter/reaktionsfenster.json")
standardabweichung = loadJson("analyseparameter/standardabweichung.json")
assetMeta = loadJson("metadata/assetMeta.json")
marketReturnsTrendsLookUp = loadJson("metadata/marketReturnsTrends.json")

analysisParameterPERYEAR = loadJson("analyseparameter/analysisParameterPERYEAR.json")

In [5]:
# FUNKTIONEN FÜR BEIDE ANSÄTZE ZUR BERECHNUNG DER TECHNISCHEN PREISNIVEAUS

In [6]:
def fibonacciLevels(df):
    
    low = df['Low'].min() 
    high = df['High'].max()
    priceRange = high - low
    fibRatios = [0.236, 0.382, 0.5, 0.618, 0.786]
    levels = {}

    levels[0.0] = low
    for r in fibRatios:
        levels[r] = low + priceRange * r
    levels[1.0] = high
    
    return levels

def supportResistance(df, order=3, tolerance=0.03, n_min=3):
    lows = df["Low"].values
    highs = df["High"].values
    pivotHighs = []
    pivotLows = []

    for i in range(order, len(df) - order):
        high = highs[i]
        low = lows[i]
        isPivotHigh = True
        isPivotLow = True

        for j in range(1, order + 1):
            if high <= highs[i - j] or high <= highs[i + j]:
                isPivotHigh = False
            if low >= lows[i - j] or low >= lows[i + j]:
                isPivotLow = False

        if isPivotHigh:
            pivotHighs.append(high)
        if isPivotLow:
            pivotLows.append(low)

    def clusterLevels(levels, tolerance, n_min):
        if len(levels) == 0:
           return []

        levels = sorted(levels)
        clusters = [[levels[0]]]

        for lvl in levels[1:]:
            currentCluster = clusters[-1]
            center = np.median(currentCluster)

            if abs(lvl - center) / center <= tolerance:
                currentCluster.append(lvl)
            else:
                clusters.append([lvl])

        validClusters = [cluster for cluster in clusters if len(cluster) >= n_min]

        mergedLevels = [float(np.median(cluster)) for cluster in validClusters]
        return mergedLevels

    support = clusterLevels(pivotLows, tolerance, n_min)
    resistance = clusterLevels(pivotHighs, tolerance, n_min)

    return support, resistance

In [7]:
def prepareFibonacciLevels(fibLevels):
    eventLevels = {}

    for ratio, value in fibLevels.items():
        if ratio in [0.0, 1.0]:
            continue
        eventLevels[f"fib_{ratio}"] = float(value)

    return eventLevels

def prepareSRLevels(support, resistance):
    eventLevels = {}

    for i, value in enumerate(support):
        eventLevels[f"support_{i+1}"] = float(value)

    for i, value in enumerate(resistance):
        eventLevels[f"resistance_{i+1}"] = float(value)

    return eventLevels

In [8]:
# EVENTLOGIK

In [ ]:
def detectEvents(df, levels, asset, year, yearlyAbsReturn, marketPhase, tauValue, windowValue, sigmaValue, assetType, market, currency, methodGroup):
    
    events = []

    highs = df["High"].values
    lows = df["Low"].values
    closes = df["Close"].values
    dates = df.index

    blockedUntil = {label: -1 for label in levels.keys()}

    for levelLabel, levelValue in levels.items():

        for t in range(len(df)):

            if t <= blockedUntil[levelLabel]:
                continue

            endT = t + windowValue
            if endT >= len(df):
                continue

            if lows[t] <= levelValue <= highs[t]:

                eventClose = closes[t]

                futureCloses = closes[t + 1:endT + 1]

                returns = (futureCloses - eventClose) / eventClose
                absReturns = np.abs(returns) 

                zValues = returns / sigmaValue
                absZValues = np.abs(zValues)

                reaction = int(np.any(absZValues >= tauValue))

                maxReturn = float(np.max(absReturns))
                maxZ = float(np.max(absZValues))

                if levelLabel.startswith("fib_"):
                    levelSubtype = "fib"
                elif levelLabel.startswith("support_"):
                    levelSubtype = "support"
                elif levelLabel.startswith("resistance_"):
                    levelSubtype = "resistance"

                events.append({
                    "asset": asset,
                    "year": year,
                    "assetType": assetType,
                    "market": market,
                    "currency": currency,
                    "marketphase": marketPhase,
                    "yearlyAbsReturn": yearlyAbsReturn,
                    "methodGroup": methodGroup,
                    "levelSubtype": levelSubtype,
                    "levelLabel": levelLabel,
                    "levelValue": float(levelValue),
                    "eventIndex": int(t),
                    "eventDate": dates[t],
                    "eventClose": float(eventClose),
                    "reaction": reaction,
                    "maxReturn": maxReturn,
                    "maxZ": maxZ,
                    "tau": float(tauValue),
                    "window": int(windowValue),
                    "sigma": float(sigmaValue)
                })

                blockedUntil[levelLabel] = endT

    return pd.DataFrame(events)

In [10]:
# SYSTEMATISCHE AUSWERTUNG - HAUPTANALYSE (GLOBAL ANALYSEPARAMETER)

In [11]:
def analyseLoopGlobalParameter(dictionaryAssetYearData):
    ergebnisseGesamtAnalyse = pd.DataFrame()
    assets = dictionaryAssetYearData.keys()

    for asset in assets:
        
        # GLOBAL Analyseparameter pro Asset
        tauValue = reaktionsschwelle[asset]
        windowValue = reaktionsfenster[asset]
        sigmaValue = standardabweichung[asset]

        assetType = assetMeta[asset]["assetType"]
        market = assetMeta[asset]["market"]
        currency = assetMeta[asset]["currency"]

        for year, df in dictionaryAssetYearData[asset].items():

            # Marketphasen Klassifikation
            yearlyAbsReturn = marketReturnsTrendsLookUp[asset][str(year)]["absReturn"]
            marketPhase = marketReturnsTrendsLookUp[asset][str(year)]["phase"]

            # Level berechnen
            fibLevelsRaw = fibonacciLevels(df)
            fibEventLevels = prepareFibonacciLevels(fibLevelsRaw)

            support, resistance = supportResistance(df, order=3, tolerance=0.03, n_min=3)
            srEventLevels = prepareSRLevels(support, resistance)

            # Fibonacci Events
            eventsFib = detectEvents(
                df=df,
                levels=fibEventLevels,
                asset=asset,
                year=year,
                yearlyAbsReturn = yearlyAbsReturn,
                marketPhase = marketPhase,
                tauValue=tauValue,
                windowValue=windowValue,
                sigmaValue=sigmaValue,
                assetType = assetType,
                market = market,
                currency = currency,
                methodGroup="fibonacci"
            )

            # Support / Resistance Events
            eventsSR = detectEvents(
                df=df,
                levels=srEventLevels,
                asset=asset,
                year=year,
                yearlyAbsReturn = yearlyAbsReturn,
                marketPhase = marketPhase,
                tauValue=tauValue,
                windowValue=windowValue,
                sigmaValue=sigmaValue,
                assetType = assetType,
                market = market,
                currency = currency,
                methodGroup="support_resistance"
            )

            eventsAll = pd.concat([eventsFib, eventsSR], ignore_index=True)

            ergebnisseGesamtAnalyse = pd.concat(
                    [ergebnisseGesamtAnalyse, eventsAll],
                    ignore_index=True
                )
    return ergebnisseGesamtAnalyse

ergebnisseGesamtAnalyseGLOBALPARAMETER = analyseLoopGlobalParameter(dictionaryAssetYearData)
# ergebnisseGesamtAnalyseGLOBALPARAMETER.to_csv("ergebnisse/ergebnisseAnalyseGLOBAL.csv", index=False)

In [12]:
ergebnisseGesamtAnalyseGLOBALPARAMETER

,asset,year,assetType,market,currency,marketphase,yearlyAbsReturn,methodGroup,levelSubtype,levelLabel,levelValue,eventIndex,eventDate,eventClose,reaction,maxReturn,maxZ,tau,window,sigma
0,AAPL,2010,Stock,USA,USD,trend,0.507219,fibonacci,fib,fib_0.236,6.665054,44,2010-03-09,6.682348,0,0.016053,0.903659,2.007222,6,0.017764
1,AAPL,2010,Stock,USA,USD,trend,0.507219,fibonacci,fib,fib_0.236,6.665054,52,2010-03-19,6.659279,1,0.045625,2.568321,2.007222,6,0.017764
2,AAPL,2010,Stock,USA,USD,trend,0.507219,fibonacci,fib,fib_0.236,6.665054,85,2010-05-06,7.378393,1,0.064324,3.620985,2.007222,6,0.017764
3,AAPL,2010,Stock,USA,USD,trend,0.507219,fibonacci,fib,fib_0.382,7.261792,67,2010-04-12,7.259734,0,0.027364,1.540412,2.007222,6,0.017764
4,AAPL,2010,Stock,USA,USD,trend,0.507219,fibonacci,fib,fib_0.382,7.261792,85,2010-05-06,7.378393,1,0.064324,3.620985,2.007222,6,0.017764
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8300,XOM,2023,Stock,USA,USD,sideways,0.029260,support_resistance,resistance,resistance_2,102.426813,198,2023-10-17,102.279068,1,0.034025,2.133119,2.067353,7,0.015951
8301,XOM,2023,Stock,USA,USD,sideways,0.029260,support_resistance,resistance,resistance_3,107.189867,27,2023-02-10,106.777702,1,0.072129,4.522015,2.067353,7,0.015951
8302,XOM,2023,Stock,USA,USD,sideways,0.029260,support_resistance,resistance,resistance_3,107.189867,76,2023-04-24,106.723541,1,0.086887,5.447229,2.067353,7,0.015951
8303,XOM,2023,Stock,USA,USD,sideways,0.029260,support_resistance,resistance,resistance_3,107.189867,173,2023-09-12,107.880135,0,0.023236,1.456747,2.067353,7,0.015951


In [13]:
# SYSTEMATISCHE AUSWERTUNG - JAHRES ANALYSEPARAMTER

In [14]:
def analyseLoopYearlyParameter(dictionaryAssetYearData):
    ergebnisseGesamtAnalyse = pd.DataFrame()
    assets = dictionaryAssetYearData.keys()

    for asset in assets:
        
        assetType = assetMeta[asset]["assetType"]
        market = assetMeta[asset]["market"]
        currency = assetMeta[asset]["currency"]

        for year, df in dictionaryAssetYearData[asset].items():

            # PARAMETER PRO JAHR PRO ASSET
            tauValue = analysisParameterPERYEAR[asset][str(year)]["tau"]
            windowValue = analysisParameterPERYEAR[asset][str(year)]["window"]
            sigmaValue = analysisParameterPERYEAR[asset][str(year)]["sigma"]

            # Marketphasen Klassifikation
            yearlyAbsReturn = marketReturnsTrendsLookUp[asset][str(year)]["absReturn"]
            marketPhase = marketReturnsTrendsLookUp[asset][str(year)]["phase"]

            # Level berechnen
            fibLevelsRaw = fibonacciLevels(df)
            fibEventLevels = prepareFibonacciLevels(fibLevelsRaw)

            support, resistance = supportResistance(df, order=3, tolerance=0.03, n_min=3)
            srEventLevels = prepareSRLevels(support, resistance)

            # Fibonacci Events
            eventsFib = detectEvents(
                df=df,
                levels=fibEventLevels,
                asset=asset,
                year=year,
                yearlyAbsReturn = yearlyAbsReturn,
                marketPhase = marketPhase,
                tauValue=tauValue,
                windowValue=windowValue,
                sigmaValue=sigmaValue,
                assetType = assetType,
                market = market,
                currency = currency,
                methodGroup="fibonacci"
            )

            # Support / Resistance Events
            eventsSR = detectEvents(
                df=df,
                levels=srEventLevels,
                asset=asset,
                year=year,
                yearlyAbsReturn = yearlyAbsReturn,
                marketPhase = marketPhase,
                tauValue=tauValue,
                windowValue=windowValue,
                sigmaValue=sigmaValue,
                assetType = assetType,
                market = market,
                currency = currency,
                methodGroup="support_resistance"
            )

            eventsAll = pd.concat([eventsFib, eventsSR], ignore_index=True)

            ergebnisseGesamtAnalyse = pd.concat(
                    [ergebnisseGesamtAnalyse, eventsAll],
                    ignore_index=True
                )
    return ergebnisseGesamtAnalyse

ergebnisseGesamtAnalyseJAHRESPARAMETER = analyseLoopYearlyParameter(dictionaryAssetYearData)
# ergebnisseGesamtAnalyseJAHRESPARAMETER.to_csv("ergebnisse/ergebnisseAnalyseJAHRES.csv", index=False)

In [15]:
ergebnisseGesamtAnalyseJAHRESPARAMETER

,asset,year,assetType,market,currency,marketphase,yearlyAbsReturn,methodGroup,levelSubtype,levelLabel,levelValue,eventIndex,eventDate,eventClose,reaction,maxReturn,maxZ,tau,window,sigma
0,AAPL,2010,Stock,USA,USD,trend,0.507219,fibonacci,fib,fib_0.236,6.665054,44,2010-03-09,6.682348,0,0.016053,0.951705,2.217750,7,0.016868
1,AAPL,2010,Stock,USA,USD,trend,0.507219,fibonacci,fib,fib_0.236,6.665054,52,2010-03-19,6.659279,1,0.061192,3.627820,2.217750,7,0.016868
2,AAPL,2010,Stock,USA,USD,trend,0.507219,fibonacci,fib,fib_0.236,6.665054,85,2010-05-06,7.378393,1,0.064324,3.813506,2.217750,7,0.016868
3,AAPL,2010,Stock,USA,USD,trend,0.507219,fibonacci,fib,fib_0.382,7.261792,67,2010-04-12,7.259734,1,0.069876,4.142608,2.217750,7,0.016868
4,AAPL,2010,Stock,USA,USD,trend,0.507219,fibonacci,fib,fib_0.382,7.261792,85,2010-05-06,7.378393,1,0.064324,3.813506,2.217750,7,0.016868
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8362,XOM,2023,Stock,USA,USD,sideways,0.029260,support_resistance,resistance,resistance_3,107.189867,27,2023-02-10,106.777702,1,0.059953,3.824254,2.087868,6,0.015677
8363,XOM,2023,Stock,USA,USD,sideways,0.029260,support_resistance,resistance,resistance_3,107.189867,76,2023-04-24,106.723541,1,0.068528,4.371256,2.087868,6,0.015677
8364,XOM,2023,Stock,USA,USD,sideways,0.029260,support_resistance,resistance,resistance_3,107.189867,173,2023-09-12,107.880135,0,0.009277,0.591782,2.087868,6,0.015677
8365,XOM,2023,Stock,USA,USD,sideways,0.029260,support_resistance,resistance,resistance_3,107.189867,180,2023-09-21,105.373428,1,0.047403,3.023754,2.087868,6,0.015677
